In [1]:
import unicodedata
import numpy as np
import pandas as pd
from keras.src.utils.module_utils import tensorflow
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

C:\Users\LOQ\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [3]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [4]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [5]:
len(data)

30000

In [6]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [7]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy. <end>'

In [8]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو. <end>'

In [9]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [10]:
vocab_size=20000
max_length=50
batch_size=32

token_en=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")
token_fa=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]




In [11]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True,dropout=0.3,recurrent_dropout=0.2)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm",dropout=0.3,recurrent_dropout=0.2)
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])

atn=tf.keras.layers.Attention(name='simple_attention')
atn_out=atn([decoder_outputs,encoder_output])
concat_out=tf.keras.layers.Concatenate(name="concat_attention")([decoder_outputs,atn_out])

In [12]:
decoder_dense=Dense(vocab_size,activation="softmax",kernel_regularizer=tf.keras.regularizers.l2(1e-4))
decoder_outputs=decoder_dense(concat_out)

In [13]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 256)   │  5,120,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │  5,120,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 50, 256), │    525,312 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_attention    │ (None, None, 256) │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, None, 512) │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ simple_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 10,260,000 │ concat_attention… │
│                     │ 20000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,550,624 (82.21 MB)

 Trainable params: 21,550,624 (82.21 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
early=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [16]:
splt=int(len(en_seq)*0.9)
train_ds=tf.data.Dataset.from_tensor_slices(((en_seq[:splt],decoder_inputs_array[:splt]),decoder_targets_array[:splt])).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_seq[splt:],decoder_inputs_array[splt:]),decoder_targets_array[splt:])).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [17]:
loss_fn= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

In [18]:
model.compile(optimizer="adam",metrics=["accuracy"],loss="SparseCategoricalCrossentropy")

In [19]:
history=model.fit(train_ds,epochs=25,validation_data=val_ds,callbacks=[early],verbose=2)

Epoch 1/25
844/844 - 718s - 851ms/step - accuracy: 0.3022 - loss: 5.6339 - val_accuracy: 0.3411 - val_loss: 4.9402
Epoch 2/25
844/844 - 434s - 514ms/step - accuracy: 0.3403 - loss: 5.0869 - val_accuracy: 0.3618 - val_loss: 4.7023
Epoch 3/25
844/844 - 563s - 667ms/step - accuracy: 0.3662 - loss: 4.8071 - val_accuracy: 0.3771 - val_loss: 4.5771
Epoch 4/25
844/844 - 1309s - 2s/step - accuracy: 0.3878 - loss: 4.5952 - val_accuracy: 0.3863 - val_loss: 4.5110
Epoch 5/25
844/844 - 659s - 780ms/step - accuracy: 0.4061 - loss: 4.4319 - val_accuracy: 0.3908 - val_loss: 4.4950
Epoch 6/25
844/844 - 339s - 402ms/step - accuracy: 0.4233 - loss: 4.2953 - val_accuracy: 0.3923 - val_loss: 4.5025
Epoch 7/25
844/844 - 338s - 400ms/step - accuracy: 0.4377 - loss: 4.1807 - val_accuracy: 0.3959 - val_loss: 4.5015


In [20]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [21]:
import pickle
model.save('Translator.keras')

In [22]:
from tensorflow.keras.models import load_model
model=load_model("Translator.keras")

In [23]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [24]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [25]:
decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
enc_out_input=Input(shape=(max_length,latent_dim))
decoder_states_inputs=[decoder_state_input_h, decoder_state_input_c]

decoder_emb2=decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

atn2=atn([decoder_outputs2,enc_out_input])
concat_out2=tf.keras.layers.Concatenate()([decoder_outputs2,atn2])
decoder_outputs2=decoder_dense(concat_out2)

decoder_model=tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)



In [26]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [27]:
def translate(sentence):

    sentence=preprossing(sentence)

    seq=token_en.texts_to_sequences([sentence])
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out,h,c=encoder_model.predict(seq)
    target_seq=np.array([[token_fa.word_index["<start>"]]])

    stop=False
    decoded= ""

    while not stop:

        output_tokens, h, c=decoder_model.predict([target_seq,h,c,enc_out])
        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word== "<end>" or len(decoded.split())>max_length:
            stop=True
        else:
            decoded+= " " +sampled_word

        target_seq=np.array([[sampled_token_index]])
        states=[h, c]

    return decoded


In [28]:
print(translate("I love you"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
 من <unk>


In [29]:
print(translate('you can'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
 تو مي خواي


In [ ]:
import sacrebleu

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(token_fa.index_word[1])
print(token_fa.index_word[2])
print(token_fa.index_word[3])
print(token_fa.index_word[4])
print(token_fa.index_word[5])